# Note this carefully.

The IterativeImputer from scikit-learn is designed primarily for numerical data, as it uses linear regression models to predict missing values. It cannot directly handle categorical data. If you have categorical data that needs to be imputed, you have a few options:

1) `Separate Numerical and Categorical Data`: Impute numerical and categorical data separately using appropriate techniques for each.
2) `Encode Categorical Data`: Convert categorical data to numerical form using techniques like one-hot encoding or label encoding, then apply imputation.
3) `Use a Different Imputer for Categorical Data`: Use imputers that can handle categorical data, like SimpleImputer with a strategy suitable for categorical data (e.g., 'most_frequent').

Here's an example of how to handle this by separating and encoding the data:

* `Step-by-Step Example`:
1. Separate Numerical and Categorical Data:
2. Encode Categorical Data:
3. Impute Missing Values:
4. Combine Data:


# We can try `IterativeImputer` on any dataset to fill missing values, but one assumption is, it gives best results in such dataset where missing column has some relation with the other columns.

See the working of Iterative imputer in the book.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [2]:
df = pd.read_csv("00 startups.csv", usecols = ['R&D Spend', 'Administration', 'Marketing Spend', 'Profit'])

In [3]:
df.head()

,R&D Spend,Administration,Marketing Spend,Profit
0,165349.20,136897.80,471784.10,192261.83
1,162597.70,151377.59,443898.53,191792.06
2,153441.51,101145.55,407934.54,191050.39
3,144372.41,118671.85,383199.62,182901.99
4,142107.34,91391.77,366168.42,166187.94


In [4]:
df = np.round(df/10000)

In [5]:
df = df.sample(5)

In [6]:
df

,R&D Spend,Administration,Marketing Spend,Profit
10,10.0,11.0,23.0,15.0
48,0.0,5.0,0.0,4.0
43,2.0,13.0,4.0,7.0
15,11.0,12.0,26.0,13.0
5,13.0,10.0,36.0,16.0


In [7]:
# Seperating Target variable

df = df.iloc[:, 0:-1]

In [8]:
df

,R&D Spend,Administration,Marketing Spend
10,10.0,11.0,23.0
48,0.0,5.0,0.0
43,2.0,13.0,4.0
15,11.0,12.0,26.0
5,13.0,10.0,36.0


In [9]:
print(df.iloc[1, 0])
print(df.iloc[3, 1])
print(df.iloc[-1, -1])

0.0
12.0
36.0


In [10]:
# Change above values to np.nan

df.iloc[1, 0] = np.nan
df.iloc[3, 1] = np.nan
df.iloc[-1, -1] = np.nan

C:\Users\jugal\AppData\Local\Temp\ipykernel_11292\3847500361.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1, 0] = np.nan
C:\Users\jugal\AppData\Local\Temp\ipykernel_11292\3847500361.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3, 1] = np.nan
C:\Users\jugal\AppData\Local\Temp\ipykernel_11292\3847500361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1, -1] = np.nan


In [11]:
df

,R&D Spend,Administration,Marketing Spend
10,10.0,11.0,23.0
48,NaN,5.0,0.0
43,2.0,13.0,4.0
15,11.0,NaN,26.0
5,13.0,10.0,NaN


# Iteration: 0 = df_0

In [12]:
# Step 1: Impute all missing values with the mean of respective columns.

mean_rd = df['R&D Spend'].mean()
mean_add = df['Administration'].mean()
mean_market = df['Marketing Spend'].mean()

df_0 = pd.DataFrame()

df_0['R&D Spend'] = df['R&D Spend'].fillna(mean_rd)
df_0['Administration'] = df['Administration'].fillna(mean_add)
df_0['Marketing Spend'] = df['Marketing Spend'].fillna(mean_market)

In [13]:
# Iteration 0
df_0

,R&D Spend,Administration,Marketing Spend
10,10.0,11.00,23.00
48,9.0,5.00,0.00
43,2.0,13.00,4.00
15,11.0,9.75,26.00
5,13.0,10.00,13.25


# Iteration 1 = df_1

In [14]:
df_1 = df_0.copy()

In [15]:
df_1

,R&D Spend,Administration,Marketing Spend
10,10.0,11.00,23.00
48,9.0,5.00,0.00
43,2.0,13.00,4.00
15,11.0,9.75,26.00
5,13.0,10.00,13.25


In [16]:
# Converting R*D Spend into np.nan.

df_1.iloc[1, 0]

9.0

In [17]:
df_1.iloc[1, 0] = np.nan

df_1

,R&D Spend,Administration,Marketing Spend
10,10.0,11.00,23.00
48,NaN,5.00,0.00
43,2.0,13.00,4.00
15,11.0,9.75,26.00
5,13.0,10.00,13.25


In [18]:
# Use first three rows to train a model and missing row for testing
# 'Administration' and 'Marketing Spend' for predictor variable and 'R&D Spend' as target variable.

x_1 = df_1.iloc[[0, 2, 3, 4], [1, 2]]    # Administration and Marketing Spend as predictor variables
y_1 = df_1.iloc[[0, 2, 3, 4], 0]         # Column R&D Spend as Class variable

pred_x_1 = df_1.iloc[1, 1:3]               # Values to be predicted.

In [19]:
x_1

,Administration,Marketing Spend
10,11.00,23.00
43,13.00,4.00
15,9.75,26.00
5,10.00,13.25


In [20]:
y_1

10    10.0
43     2.0
15    11.0
5     13.0
Name: R&D Spend, dtype: float64

In [21]:
pred_x_1

Administration     5.0
Marketing Spend    0.0
Name: 48, dtype: float64

In [22]:
lr = LinearRegression()

In [23]:
lr.fit(x_1, y_1)

LinearRegression()

In [24]:
lr.predict([pred_x_1])

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([29.53654685])

In [25]:
a = pred_x_1.values.reshape(1, 2)

In [26]:
pred_value = lr.predict(a)

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [27]:
pred_value

array([29.53654685])

In [28]:
# Filling missing value of first column with this predicted calue

df_1.iloc[1, 0] = pred_value.round(2)

In [29]:
df_1

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,9.75,26.00
5,13.00,10.00,13.25


In [30]:
# Removing Administration's mean imputed value.

df_1.iloc[3,1]

9.75

In [31]:
df_1.iloc[3,1] = np.nan

In [32]:
df_1

,R&D Spend,Administration,Marketing Spend
10,10.00,11.0,23.00
48,29.54,5.0,0.00
43,2.00,13.0,4.00
15,11.00,NaN,26.00
5,13.00,10.0,13.25


In [33]:
x_1 = df_1.iloc[[0, 1, 2, 4], [0, 2]]    # 'Marketing Spend' and 'R&D Spend' as Predictor variable.
y_1 = df_1.iloc[[0,1, 2, 4], [1]]        # 'Administration' as Target variable.

pred_x_1 = df_1.iloc[3, [0, 2]].values.reshape(1, 2)

In [34]:
x_1

,R&D Spend,Marketing Spend
10,10.00,23.00
48,29.54,0.00
43,2.00,4.00
5,13.00,13.25


In [35]:
y_1

,Administration
10,11.0
48,5.0
43,13.0
5,10.0


In [36]:
pred_x_1

array([[11., 26.]])

In [37]:
# Training the model

lr.fit(x_1, y_1)

LinearRegression()

In [38]:
# Predict.

pred_value = lr.predict(pred_x_1)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([[10.76891413]])

In [39]:
# Imputing predicted value to missing value.

df_1.iloc[3, 1]

nan

In [40]:
df_1.iloc[3, 1] = pred_value.round(2)

In [41]:
df_1

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.77,26.00
5,13.00,10.00,13.25


In [42]:
# convert column 3 mean imputed value as np.nan

df_1.iloc[4, 2]

13.25

In [43]:
df_1.iloc[4, 2] = np.nan

In [44]:
df_1

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.0
48,29.54,5.00,0.0
43,2.00,13.00,4.0
15,11.00,10.77,26.0
5,13.00,10.00,NaN


In [45]:
# Spliting to x and y.

x_1 = df_1.iloc[[0, 1, 2, 3], [0, 1]]    # 'R*D Spend' and 'Administration' as predictor variable and
y_1 = df_1.iloc[[0, 1, 2, 3], 2]         # 'Marketing Speed' as Target variable.

pred_x_1 = df_1.iloc[4, [0, 1]].values.reshape(1, 2)

In [46]:
x_1

,R&D Spend,Administration
10,10.00,11.00
48,29.54,5.00
43,2.00,13.00
15,11.00,10.77


In [47]:
y_1

10    23.0
48     0.0
43     4.0
15    26.0
Name: Marketing Spend, dtype: float64

In [48]:
pred_x_1

array([[13., 10.]])

In [49]:
# Training the model

lr.fit(x_1, y_1)

LinearRegression()

In [50]:
# Making prediction
pred_value = lr.predict(pred_x_1)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([14.3891371])

In [51]:
# Imputing missing value with predicted value.
df_1.iloc[4, 2]

nan

In [52]:
df_1.iloc[4, 2] = pred_value.round(2)

In [53]:
df_1

# Here, we completed our first iteration.

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.77,26.00
5,13.00,10.00,14.39


In [54]:
# Subtract 0th iteration from 1st iteration

df_1 - df_0

# The difference should be almost zero.

,R&D Spend,Administration,Marketing Spend
10,0.00,0.00,0.00
48,20.54,0.00,0.00
43,0.00,0.00,0.00
15,0.00,1.02,0.00
5,0.00,0.00,1.14


# Iteration 2 = df_2

In [55]:
df_2 = df_1.copy()

In [56]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.77,26.00
5,13.00,10.00,14.39


In [57]:
# Changing R&D's row to np.nan.

df_2.iloc[1,0]

29.54

In [58]:
df_2.iloc[1, 0] = np.nan

In [59]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.0,11.00,23.00
48,NaN,5.00,0.00
43,2.0,13.00,4.00
15,11.0,10.77,26.00
5,13.0,10.00,14.39


In [60]:
# Spliting into x and y.

x_2 = df_2.iloc[[0, 2, 3, 4], 1:3]    # 'Administration' and 'Marketing Spend' as predictor variable
y_2 = df_2.iloc[[0, 2, 3, 4], 0]      # 'R&D Spend' as target variable.

pred_x_2 = df_2.iloc[1, 1:3].values.reshape(1, 2)    # Missing value to be predicted.

In [61]:
x_2

,Administration,Marketing Spend
10,11.00,23.00
43,13.00,4.00
15,10.77,26.00
5,10.00,14.39


In [62]:
y_2

10    10.0
43     2.0
15    11.0
5     13.0
Name: R&D Spend, dtype: float64

In [63]:
pred_x_2

array([[5., 0.]])

In [64]:
# Training the model

lr.fit(x_2, y_2)

LinearRegression()

In [65]:
# Making predicting.

pred_value = lr.predict(pred_x_2)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([29.53632507])

In [66]:
# Filling missing value with the predicted value.

df_2.iloc[1, 0]

nan

In [67]:
df_2.iloc[1, 0] = pred_value.round(2)

In [68]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.77,26.00
5,13.00,10.00,14.39


In [69]:
# Converting 'Administration's column as np.nan.

df_2.iloc[-2, -2]

10.77

In [70]:
df_2.iloc[-2, -2] = np.nan

In [71]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.0,23.00
48,29.54,5.0,0.00
43,2.00,13.0,4.00
15,11.00,NaN,26.00
5,13.00,10.0,14.39


In [72]:
# Spliting into x and y.

x_2 = df_2.iloc[[0, 1, 2, 4], [0, 2]]    # 'R*D Spend' and 'Marketing Spend' as Predictor variable and
y_2 = df_2.iloc[[0, 1, 2, 4], 1]         # 'Administration' as class variable

pred_x_2 = df_2.iloc[3, [0, 2]]          # Missing value to be predicted.

In [73]:
x_2

,R&D Spend,Marketing Spend
10,10.00,23.00
48,29.54,0.00
43,2.00,4.00
5,13.00,14.39


In [74]:
y_2

10    11.0
48     5.0
43    13.0
5     10.0
Name: Administration, dtype: float64

In [75]:
pred_x_2

R&D Spend          11.0
Marketing Spend    26.0
Name: 15, dtype: float64

In [76]:
# Training the model

lr.fit(x_2, y_2)

LinearRegression()

In [77]:
# MAking prediction

pred_value = lr.predict(pred_x_2.values.reshape(1, 2))
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([10.76131338])

In [78]:
# Filling missing value with the predicted value.

df_2.iloc[-2,  -2]

nan

In [79]:
df_2.iloc[-2,  -2] = pred_value.round(2)

In [80]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.39


In [81]:
# Changing 'Marketing Spend's column as np.nan

df_2.iloc[-1, -1]

14.39

In [82]:
df_2.iloc[-1, -1] = np.nan

In [83]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.0
48,29.54,5.00,0.0
43,2.00,13.00,4.0
15,11.00,10.76,26.0
5,13.00,10.00,NaN


In [84]:
# Spliting into x and y

x_2 = df_2.iloc[[0, 1, 2, 3], [0, 1]]    # 'R*D Spend ' and 'Administration' as predictor variable and
y_2 = df_2.iloc[[0, 1, 2, 3], -1]        # 'Marketing Spend' as class varibale

pred_x_2 = df_1.iloc[-1, [0, 1]].values.reshape(1, 2)    # Missing value to be predicted.

In [85]:
x_2

,R&D Spend,Administration
10,10.00,11.00
48,29.54,5.00
43,2.00,13.00
15,11.00,10.76


In [86]:
y_2

10    23.0
48     0.0
43     4.0
15    26.0
Name: Marketing Spend, dtype: float64

In [87]:
pred_x_2

array([[13., 10.]])

In [88]:
# Training the model

lr.fit(x_2, y_2)

LinearRegression()

In [89]:
# MAking prediction

pred_value = lr.predict(pred_x_2)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([14.56349366])

In [90]:
# Filling missing value with the predicted value.

df_2.iloc[-1, -1]

nan

In [91]:
df_2.iloc[-1, -1] = pred_value.round(2)

In [92]:
df_2

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.56


In [93]:
# Difference between 2nd and 1st iteration.

df_2 - df_1

,R&D Spend,Administration,Marketing Spend
10,0.0,0.00,0.00
48,0.0,0.00,0.00
43,0.0,0.00,0.00
15,0.0,-0.01,0.00
5,0.0,0.00,0.17


# Iteration 3 = df_3

In [94]:
df_3 = df_2.copy()

In [95]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.56


In [96]:
# Converting R&D Spend's column as np.nan

df_3.iloc[1, 0]

29.54

In [97]:
df_3.iloc[1, 0] = np.nan

In [98]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.0,11.00,23.00
48,NaN,5.00,0.00
43,2.0,13.00,4.00
15,11.0,10.76,26.00
5,13.0,10.00,14.56


In [99]:
# Spliting into x and y

x_3 = df_3.iloc[[0, 2, 3, 4], [1, 2]]    # 'Administration' and 'MArketing Spend' as Predictor variable and
y_3 = df_3.iloc[[0, 2, 3, 4], 0]           # 'R&D Spend' as Class variable

pred_x_3 = df.iloc[1, [1, 2]].values.reshape(1, 2)    # Missing value to be predicted.

In [100]:
x_3

,Administration,Marketing Spend
10,11.00,23.00
43,13.00,4.00
15,10.76,26.00
5,10.00,14.56


In [101]:
y_3

10    10.0
43     2.0
15    11.0
5     13.0
Name: R&D Spend, dtype: float64

In [102]:
pred_x_3

array([[5., 0.]])

In [103]:
# Trainning the model

lr.fit(x_3, y_3)

LinearRegression()

In [104]:
# MAking Prediction

pred_value = lr.predict(pred_x_3)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([29.54087099])

In [105]:
# Filling missing value with the predicted value

df_3.iloc[1, 0]

nan

In [106]:
df_3.iloc[1, 0] = pred_value.round(2)

In [107]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.56


In [108]:
# Converting Administratin's column to np.nan

df_3.iloc[3, 1]

10.76

In [109]:
df_3.iloc[3, 1] = np.nan

In [110]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.0,23.00
48,29.54,5.0,0.00
43,2.00,13.0,4.00
15,11.00,NaN,26.00
5,13.00,10.0,14.56


In [111]:
# Spliting into x and y

x_3 = df_3.iloc[[0, 1, 2, 4], [0, 2]]    # 'R&D Spend' and 'Marketing Spend' as predictor variables and 
y_3 = df_3.iloc[[0, 1, 2, 4], 1]         # 'Administration' as Class variable

pred_x_3 = df_3.iloc[-2, [0, 2]].values.reshape(1, 2)           # Missing value to be predicted.

In [112]:
x_3

,R&D Spend,Marketing Spend
10,10.00,23.00
48,29.54,0.00
43,2.00,4.00
5,13.00,14.56


In [113]:
y_3

10    11.0
48     5.0
43    13.0
5     10.0
Name: Administration, dtype: float64

In [114]:
pred_x_3

array([[11., 26.]])

In [115]:
# Training the model

lr.fit(x_3, y_3)

LinearRegression()

In [116]:
# Making prediction

pred_value = lr.predict(pred_x_3)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([10.76006106])

In [117]:
#Filling missing value t=with the predicted value

df_3.iloc[-2, -2]

nan

In [118]:
df_3.iloc[-2, -2] = pred_value.round(2)

In [119]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.56


In [120]:
# Converting Marketing Spend column into np.nan

df_3.iloc[-1, -1]

14.56

In [121]:
df_3.iloc[-1, -1] = np.nan

In [122]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.0
48,29.54,5.00,0.0
43,2.00,13.00,4.0
15,11.00,10.76,26.0
5,13.00,10.00,NaN


In [123]:
# Spliting into x and y

x_3 = df_3.iloc[0:4, [0, 1]]    # 'R*D Spend' and 'Administration' as Predictor varables and
y_3 = df_3.iloc[0:4, -1]        # 'Marketing Spend' as Class variable

pred_x_3 = df_3.iloc[-1, [0, 1]].values.reshape(1, 2)

In [124]:
x_3

,R&D Spend,Administration
10,10.00,11.00
48,29.54,5.00
43,2.00,13.00
15,11.00,10.76


In [125]:
y_3

10    23.0
48     0.0
43     4.0
15    26.0
Name: Marketing Spend, dtype: float64

In [126]:
pred_x_3

array([[13., 10.]])

In [127]:
# Training the model

lr.fit(x_3, y_3)

LinearRegression()

In [128]:
# MAking prediction

pred_value = lr.predict(pred_x_3)
pred_value

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([14.56349366])

In [129]:
# Filling missing value with the predict value.

df_3.iloc[-1, -1]

nan

In [130]:
df_3.iloc[-1, -1] = pred_value.round(2)

In [131]:
df_3

,R&D Spend,Administration,Marketing Spend
10,10.00,11.00,23.00
48,29.54,5.00,0.00
43,2.00,13.00,4.00
15,11.00,10.76,26.00
5,13.00,10.00,14.56


In [132]:
# Difference between 3rd and 2nd Iteration

df_3 - df_2

,R&D Spend,Administration,Marketing Spend
10,0.0,0.0,0.0
48,0.0,0.0,0.0
43,0.0,0.0,0.0
15,0.0,0.0,0.0
5,0.0,0.0,0.0


In [133]:
df_3.sub(df_2)

,R&D Spend,Administration,Marketing Spend
10,0.0,0.0,0.0
48,0.0,0.0,0.0
43,0.0,0.0,0.0
15,0.0,0.0,0.0
5,0.0,0.0,0.0


# Next: Created a function named `iterative_imputation()` where performed all these steps inside a loop.